# Lab 4 — 멀티에이전트 · RAG · 평가

> **이론 복습 — Session 4 슬라이드**
> - 멀티에이전트: 오케스트레이터가 전문 서브에이전트로 **라우팅**
> - RAG: 문서를 **검색**해 근거로 답한다 (검색도 하나의 도구)
> - 평가: **Scripted LLM** 으로 결정론적 테스트

## 학습 목표
1. `search_docs` 도구로 **RAG 에이전트**를 만든다
2. 질문을 분류해 서브에이전트로 보내는 **오케스트레이터**를 만든다
3. `ScriptedLLM` 으로 에이전트를 **결정론적으로 테스트**한다


## 0. 준비
`labs/` 폴더에서 실행하세요. `Agent` 클래스는 Lab 3의 완성본을
`common/agent.py` 에서 가져옵니다.


In [ ]:
import json
import re
from common.llm import LLMClient, ScriptedLLM, LLMResponse, ToolCall
from common.agent import Agent
from common import tools, mini_sim

## 1. RAG — `search_docs` 도구 만들기

`labs/data/knowledge.json` 에 두 연구 프로젝트와 에이전트 개념을 설명한
짧은 문서 청크가 들어 있습니다. 질문과 관련된 청크를 찾아 주는 검색 도구를 만듭니다.


In [ ]:
KNOWLEDGE = json.load(open("data/knowledge.json", encoding="utf-8"))
print(f"{len(KNOWLEDGE)} knowledge chunks loaded.")


def _terms(text):
    """Split text into lower-cased terms of length >= 2."""
    return [t for t in re.findall(r"[a-z0-9가-힣]+", text.lower()) if len(t) >= 2]


def search_docs(query, k=3):
    """Return the k knowledge chunks most relevant to the query."""
    q = query.lower()
    scored = []
    for chunk in KNOWLEDGE:
        key_terms = _terms(chunk["id"] + " " + chunk["title"])
        score = sum(1 for t in key_terms if t in q)
        if score:
            scored.append((score, chunk))
    scored.sort(key=lambda pair: -pair[0])
    return [{"title": c["title"], "text": c["text"]} for _, c in scored[:k]]


# try it directly
for hit in search_docs("MCP가 뭐야?"):
    print("-", hit["title"])

검색이 동작하니, 이제 이걸 **도구**로 만들어 에이전트에 붙입니다.
RAG는 별도의 거창한 구조가 아니라 — *그냥 검색 도구를 가진 에이전트* 입니다.


In [ ]:
SCHEMA_SEARCH_DOCS = {
    "name": "search_docs",
    "description": "Search the project knowledge base. Use this whenever asked "
                   "about DTUMOS, mega-region-ai, or any agentic-AI concept.",
    "parameters": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "The search query."}},
        "required": ["query"],
    },
}

rag_agent = Agent(
    llm=LLMClient(),
    schemas=[SCHEMA_SEARCH_DOCS],
    functions={"search_docs": search_docs},
    system=("You are a helpful tutor. To answer questions about the projects "
            "or about agentic AI, FIRST call search_docs, then answer in "
            "Korean based on what it returns."),
)
print(rag_agent.run("DTUMOS가 뭐야?"))

## 2. 오케스트레이터 — 라우팅

질문 종류에 따라 알맞은 **서브에이전트**로 보냅니다. 서브에이전트는 그냥
각자 도구가 다른 `Agent` 일 뿐입니다.


In [ ]:
# sub-agent A — commute data analysis
analysis_agent = Agent(
    LLMClient(verbose=False), tools.get_schemas(), dict(tools.TOOLBOX),
    system="You are a commute-data analyst. Answer in Korean.",
)

# sub-agent B — taxi simulation
SIM_SCHEMA = {
    "name": "run_mini_simulation",
    "description": "Run a taxi-dispatch simulation; returns service metrics.",
    "parameters": {"type": "object", "properties": {
        "fleet_size": {"type": "integer", "description": "Number of taxis."},
        "dispatch": {"type": "string", "enum": ["nearest", "fifo"]},
    }, "required": []},
}
sim_agent = Agent(
    LLMClient(verbose=False), [SIM_SCHEMA],
    {"run_mini_simulation": mini_sim.run_mini_simulation},
    system="You run taxi simulations. Answer in Korean.",
)


def orchestrator(question):
    """Route a question to the right specialist sub-agent."""
    sim_words = ["시뮬", "택시", "배차", "대기시간", "fleet"]
    if any(w in question for w in sim_words):
        kind, agent = "simulation", sim_agent
    else:
        kind, agent = "analysis", analysis_agent
    print(f"[orchestrator] routed to: {kind}")
    return agent.run(question, verbose=False)


print(orchestrator("자족도가 가장 높은 생활권은?"))
print()
print(orchestrator("택시 30대로 시뮬레이션하면 평균 대기시간은?"))

## 3. 결정론적 테스트 — `ScriptedLLM`

진짜 LLM은 호출마다 답이 다르고, 비용·네트워크가 필요합니다. 그래서 테스트에는
**가짜 LLM** 을 씁니다. `ScriptedLLM` 은 미리 정한 응답을 순서대로 재생합니다.
→ 같은 입력에 늘 같은 결과 = 테스트 가능.


In [ ]:
# pretend the model: (1) calls get_top_flows, then (2) gives a final answer
scripted = ScriptedLLM([
    LLMResponse(tool_calls=[ToolCall("get_top_flows", {"n": 3})]),
    LLMResponse(text="가장 큰 통근 통행은 송파구 → 강남구입니다."),
])

test_agent = Agent(scripted, tools.get_schemas(["get_top_flows"]),
                   dict(tools.TOOLBOX))
out = test_agent.run("상위 통행은?", verbose=False)

assert "송파구" in out, f"unexpected answer: {out}"
print("✅ test passed —", out)

## 4. (선택) MCP — 도구를 표준 서버로

지금 우리 도구는 이 노트북 안에서만 쓸 수 있습니다. **MCP(Model Context
Protocol)** 로 감싸면, Claude Desktop 등 어떤 클라이언트든 같은 도구를 씁니다.

아래는 *개념 스케치* 입니다 (실행하지 않음 — `pip install mcp` 필요).

```python
# mcp_server.py  (개념 스케치)
from mcp.server import Server

server = Server("mobility-tools")

@server.tool()
def get_top_flows(n: int = 10):
    from common.tools import get_top_flows as impl
    return impl(n)

# 이제 이 서버에 연결한 어떤 에이전트도 get_top_flows 를 쓸 수 있다.
# DTUMOS 연구 프로젝트도 시뮬레이션 기능을 이렇게 MCP 서버로 제공한다.
```


## 🔧 TODO — 서브에이전트 추가하기

지금 오케스트레이터는 "분석"과 "시뮬레이션" 두 갈래뿐입니다.
세 번째로 **"문서/개념 질문"** 갈래를 추가하세요 — 이미 만든 `rag_agent` 를 쓰면 됩니다.

`smart_orchestrator` 에서 질문에 "뭐야", "설명", "개념" 같은 말이 있으면
`rag_agent` 로 보내도록 완성하세요.


In [ ]:
def smart_orchestrator(question):
    # ✅ docs 갈래를 가장 먼저 확인
    docs_words = ["뭐야", "무엇", "설명", "개념", "뜻"]
    sim_words = ["시뮬", "택시", "배차", "대기시간"]

    if any(w in question for w in docs_words):
        print("[router] -> docs (RAG)")
        return rag_agent.run(question, verbose=False)
    if any(w in question for w in sim_words):
        print("[router] -> simulation")
        return sim_agent.run(question, verbose=False)
    print("[router] -> analysis")
    return analysis_agent.run(question, verbose=False)


print(smart_orchestrator("MCP가 뭐야?"))

## 정리

- RAG = 검색 도구(`search_docs`)를 가진 에이전트 — 새 구조가 아니다
- 오케스트레이터 = 질문을 분류해 전문 서브에이전트로 **라우팅**
- `ScriptedLLM` 으로 API 없이 **결정론적 테스트** 가능
- MCP = 도구를 표준 서버로 만들어 재사용

**다음 — Session 5 (캡스톤)**: 배운 것을 모아 나만의 에이전트를 확장합니다.
